# NLP Assignment — Dataset selection, text preprocessing and N-gram language model

In [ ]:
import os, re, math, collections
import pandas as pd
import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
            'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer, PorterStemmer

## 1. Dataset selection

**AG News** (the dataset assigned by the instructor): each row has a class (1=World, 2=Sports, 3=Business, 4=Sci/Tech), a title and a description.
The text we process is **Title + Description**.

In [ ]:
# Works on Colab (asks for the files) and locally (if train.csv / test.csv are in the same folder)
if not (os.path.exists('train.csv') and os.path.exists('test.csv')):
    from google.colab import files
    files.upload()

train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')
print('train:', train_df.shape, '| test:', test_df.shape)
display(train_df.head())

class_names = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
print(train_df['Class Index'].map(class_names).value_counts())
print(test_df['Class Index'].map(class_names).value_counts())

## 2. Text preprocessing

Pipeline (in this order):

1. **Cleaning** (on the raw text, *before* lowercasing):
   * literal `\` characters (they stand for line breaks) → space
   * HTML leftovers: tags such as `lt;b gt;` / `lt;A HREF=... gt;`, entities (`quot;`, `amp;`, `#36;`) — matched with a word boundary so normal words such as `result;` or `camp;` are **not** damaged
   * URLs
   * news-agency boilerplate: `(Reuters)`, `(AP)`, `NEW YORK (Reuters) -`, `AP -` … These are not language; if we keep them, `<s> reuters` and `ap </s>` become the most frequent bigrams
   * `U.S.` → `USA` (so the sentence splitter does not cut inside it and the token is not lost)
   * hyphens → space (`short-sellers` → `short sellers`)
2. **Segmentation** – `sent_tokenize`. Done on the **original-case** text because the sentence splitter (punkt) uses capital letters and abbreviations to find sentence boundaries. We need sentences for the N-gram model (`<s>` … `</s>`).
3. **Tokenization** – `word_tokenize` for each sentence.
4. **POS tagging** – only used to lemmatize correctly (verbs vs nouns).
5. **Lowercasing**
6. **Remove punctuation and numbers** – keep only alphabetic tokens (and drop single letters such as `b`, `x`).
7. **Remove stop words** – NLTK English list + the agency names (`reuters`, `ap`, `afp`, `upi`).
8. **Lemmatization** – WordNet lemmatizer **with the POS tag** (without it every word is treated as a noun, so `announced` stays `announced`). Stemming is shown for comparison only.

In [ ]:
SOURCES = (r'(?:Reuters|AP|AFP|UPI|Associated Press|SPACE\.com|Bloomberg|Canadian Press|'
           r'AP Business|AP Sports|USATODAY\.com|Forbes\.com|PC World|IDG News Service)')

def clean_raw(text):
    t = str(text).replace('\\', ' ')                          # literal backslash = line break
    t = re.sub(r'&?\blt;.*?&?\bgt;', ' ', t)                  # HTML tags:  lt;b gt; ... lt;/b gt;
    t = re.sub(r'&?\b(?:lt|gt|quot|amp|nbsp);', ' ', t)       # leftover entities (\b => 'result;' is safe)
    t = re.sub(r'#\d+;', ' ', t)                              # #36;  (dollar sign) etc.
    t = re.sub(r'(?:https?://|www\.)\S+', ' ', t)             # URLs
    t = re.sub(r'\bhref\S*|\btarget=\S*', ' ', t)             # broken link attributes
    t = re.sub(r'\bU\.S\.A?\.?', 'USA', t)                    # U.S. -> USA
    return t

def clean_title(t):
    t = clean_raw(t).strip()
    return re.sub(r'\(\s*' + SOURCES + r'\s*\)\s*$', '', t).strip()          # "... (Reuters)"

def clean_desc(d):
    d = clean_raw(d).strip()
    d = re.sub(r'^[^-\u2013\u2014]{0,50}?\(\s*' + SOURCES + r'\s*\)\s*[-\u2013\u2014]+\s*', '', d)  # "NEW YORK (Reuters) - "
    d = re.sub(r'^' + SOURCES + r'\s*[-\u2013\u2014]+\s*', '', d)                                     # "Reuters - "
    return d.strip()

def build_text(title, desc):
    title = clean_title(title)
    sep = ' ' if title.endswith(('.', '!', '?')) else '. '
    return title + sep + clean_desc(desc)

for df in (train_df, test_df):
    df['raw_text'] = df['Title'].astype(str) + '. ' + df['Description'].astype(str)
    df['text'] = [build_text(t, d) for t, d in zip(df['Title'], df['Description'])]

print('BEFORE:', train_df['raw_text'][1], '\n')
print('AFTER :', train_df['text'][1])

### 2.1 Step-by-step on one example (so every step is visible)

In [ ]:
stop_words = set(stopwords.words('english')) | {'reuters', 'ap', 'afp', 'upi'}
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def wn_pos(tag):                                   # Penn Treebank tag -> WordNet POS
    if tag.startswith('J'): return wordnet.ADJ
    if tag.startswith('V'): return wordnet.VERB
    if tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

def fix_hyphens(t):
    return re.sub(r'\s+', ' ', re.sub(r'[-\u2013\u2014]+', ' ', t)).strip()

i = 1
print('RAW          :', test_df['raw_text'][i], '\n')
s0 = fix_hyphens(test_df['text'][i])
print('CLEANED      :', s0, '\n')

sents = sent_tokenize(s0)                          # segmentation (original case!)
print('SEGMENTATION :')
for k, s in enumerate(sents, 1): print('   ', k, '-', s)
print()

toks = [word_tokenize(s) for s in sents]
print('TOKENIZATION :')
for t in toks: print('   ', t)
print()

tagged = [nltk.pos_tag(t) for t in toks]
print('POS TAGS     :')
for t in tagged: print('   ', t)
print()

lower = [[(w.lower(), tg) for w, tg in t] for t in tagged]
print('LOWERCASE    :')
for t in lower: print('   ', [w for w, _ in t])
print()

no_punct = [[(w, tg) for w, tg in t if w.isalpha() and len(w) > 1] for t in lower]     # drops punctuation AND numbers
print('NO PUNCT/NUM :')
for t in no_punct: print('   ', [w for w, _ in t])
print()

no_stop = [[(w, tg) for w, tg in t if w not in stop_words] for t in no_punct]
print('NO STOPWORDS :')
for t in no_stop: print('   ', [w for w, _ in t])
print()

lem = [[lemmatizer.lemmatize(w, wn_pos(tg)) for w, tg in t] for t in no_stop]
stm = [[stemmer.stem(w) for w, _ in t] for t in no_stop]
print('LEMMATIZED   :')
for t in lem: print('   ', t)
print()
print('(STEMMED, for comparison):')
for t in stm: print('   ', t)

In [ ]:
def preprocess(text):
    # returns a list of sentences; each sentence is a list of clean tokens
    text = fix_hyphens(text)
    out = []
    for sent in sent_tokenize(text):                                       # segmentation
        tagged = nltk.pos_tag(word_tokenize(sent))                         # tokenization + POS
        toks = []
        for w, tg in tagged:
            w = w.lower()                                                  # lowercase
            if not w.isalpha() or len(w) < 2:                              # remove punctuation, numbers, 1-letter tokens
                continue
            if w in stop_words:                                            # remove stop words
                continue
            toks.append(lemmatizer.lemmatize(w, wn_pos(tg)))               # lemmatization (with POS)
        if toks:
            out.append(toks)
    return out

# Hold out 10% of TRAIN articles as a validation set (used only to choose K and lambda).
val_df        = train_df.sample(frac=0.10, random_state=42)
train_main_df = train_df.drop(val_df.index)

train_sents = [s for txt in train_main_df['text'] for s in preprocess(txt)]   # TRAIN      -> builds the model
val_sents   = [s for txt in val_df['text']        for s in preprocess(txt)]   # VALIDATION -> tunes K and lambda
test_sents  = [s for txt in test_df['text']       for s in preprocess(txt)]   # TEST       -> final evaluation only

for name, ss in [('train', train_sents), ('val', val_sents), ('test', test_sents)]:
    print(f'{name:5s} sentences: {len(ss):>8,} | tokens: {sum(map(len, ss)):>10,}')
print(train_sents[:3])

## 3. N-gram language model (Bigram, Markov assumption)

### Theory

The chain rule gives the exact probability of a sentence $w_1 \dots w_n$:

$$P(w_1,\dots,w_n)=\prod_{i=1}^{n}P(w_i\mid w_1,\dots,w_{i-1})$$

The **Markov assumption** (first order) says a word depends only on the previous word:

$$P(w_i\mid w_1,\dots,w_{i-1})\approx P(w_i\mid w_{i-1})$$

so

$$P(w_1,\dots,w_n)\approx\prod_{i=1}^{n+1}P(w_i\mid w_{i-1})$$

with the sentence padded as `<s> w1 ... wn </s>` (so $w_0=$ `<s>` and $w_{n+1}=$ `</s>`).

**Maximum likelihood estimate (MLE):**
$$P(w_i\mid w_{i-1})=\frac{C(w_{i-1},w_i)}{C(w_{i-1})}$$

**Add-one (Laplace) smoothing** – needed because an unseen bigram would give probability 0 for the whole sentence:
$$P_{Laplace}(w_i\mid w_{i-1})=\frac{C(w_{i-1},w_i)+1}{C(w_{i-1})+V}$$

where $V$ is the number of different tokens that can be predicted (the vocabulary + `<UNK>` + `</s>`; `<s>` is only ever a context, never predicted).

### 3.1 Build vocabulary and counts — **from TRAIN only**

In [ ]:
MIN_COUNT = 2   # words seen fewer than 2 times in train become <UNK>
word_freq = collections.Counter(w for s in train_sents for w in s)
vocab = {w for w, c in word_freq.items() if c >= MIN_COUNT} | {'<UNK>', '</s>'}
V = len(vocab)                        # predictable tokens (no <s>)
print('vocabulary size V =', V)

def prep(sent):                       # map OOV words to <UNK>, add boundary markers
    return ['<s>'] + [w if w in vocab else '<UNK>' for w in sent] + ['</s>']

unigram  = collections.Counter()      # C(prev): context counts (includes <s>)
uni_pred = collections.Counter()      # counts of predicted tokens (no <s>) -> unigram baseline
bigram   = collections.Counter()
for s in train_sents:
    p = prep(s)
    unigram.update(p)
    uni_pred.update(p[1:])
    bigram.update(zip(p[:-1], p[1:]))
N_pred = sum(uni_pred.values())

followers = collections.defaultdict(list)          # index: prev -> [(next, count)]
for (a, b), c in bigram.items():
    followers[a].append((b, c))

print('distinct bigrams in train:', len(bigram))
print('most common bigrams:', bigram.most_common(10))

In [ ]:
def p_mle(prev, w):
    return bigram[(prev, w)] / unigram[prev] if unigram[prev] else 0.0

def p_laplace(prev, w):
    return (bigram[(prev, w)] + 1) / (unigram[prev] + V)

def p_addk(prev, w, k):
    return (bigram[(prev, w)] + k) / (unigram[prev] + k * V)

def p_unigram(w):                     # add-one smoothed unigram baseline
    return (uni_pred[w] + 1) / (N_pred + V)

def p_interp(prev, w, lam):           # lam * bigram MLE + (1-lam) * unigram
    return lam * p_mle(prev, w) + (1 - lam) * p_unigram(w)

def sentence_table(sent):
    p = prep(sent)
    rows = []
    for prev, w in zip(p[:-1], p[1:]):
        c_bi, c_prev = bigram[(prev, w)], unigram[prev]
        rows.append({
            'bigram P(w | prev)': f'P({w} | {prev})',
            'C(prev,w)': c_bi, 'C(prev)': c_prev,
            'MLE': c_bi / c_prev if c_prev else 0.0,
            'Laplace = (C+1)/(C(prev)+V)': (c_bi + 1) / (c_prev + V),
        })
    return pd.DataFrame(rows)

def show_calculation(sent):
    # full worked calculation of P(sentence) under the bigram (Markov) model
    print('Sentence (after preprocessing):', sent)
    print('With <UNK> mapping + markers  :', prep(sent), '\n')
    tbl = sentence_table(sent)
    display(tbl)
    mle = tbl['MLE'].tolist()
    lap = tbl['Laplace = (C+1)/(C(prev)+V)'].tolist()
    print('\nMarkov assumption:  P(sentence) ≈ ' + ' × '.join(tbl['bigram P(w | prev)']))
    print('\nMLE      :', ' × '.join(f'{x:.3e}' for x in mle))
    print('         = ', math.prod(mle), '   <-- 0 as soon as ONE bigram is unseen')
    print('\nLaplace  :', ' × '.join(f'{x:.3e}' for x in lap))
    print('         = ', math.prod(lap))
    print('\nlog10 P (Laplace) = Σ log10(term) =', sum(math.log10(x) for x in lap),
          '  (logs avoid underflow for long sentences)')

pd.set_option('display.float_format', lambda x: f'{x:.3e}' if (x != 0 and abs(x) < 1e-3) else f'{x:.6f}')

### 3.2 Worked example 1 — a sentence from the **TEST** set (unseen by the model)

In [ ]:
for raw in test_df['text']:
    ps = preprocess(raw)
    if ps and 5 <= len(ps[0]) <= 7:
        break
print('Raw sentence :', sent_tokenize(fix_hyphens(raw))[0], '\n')
show_calculation(ps[0])

### 3.3 Worked example 2 — a sentence from the **TRAIN** set

In [ ]:
show_calculation(train_sents[0])

### 3.4 Most likely next words

In [ ]:
def next_word_probs(prev, k=8):
    cands = sorted(followers[prev], key=lambda x: -x[1])[:k]
    return pd.DataFrame([{'next word': w, 'C(prev,w)': c, 'C(prev)': unigram[prev],
                          'P(w|prev) MLE': c / unigram[prev]} for w, c in cands])

for w in ['oil', 'microsoft']:
    print(f'\nMost likely words after "{w}":')
    display(next_word_probs(w))

## 4. Evaluation — perplexity

$$PP(W)=\exp\!\Big(-\frac{1}{N}\sum_{i=1}^{N}\ln P(w_i\mid w_{i-1})\Big)$$

Lower perplexity = better model. All models are **trained on train only**.

* The hyper-parameters (`K` for add-k, `λ` for interpolation) are chosen on the **validation** set.
* The **test** set is used only once, for the final numbers.

In [ ]:
def perplexity(sents, prob_fn):
    ls, n = 0.0, 0
    for s in sents:
        p = prep(s)
        for prev, w in zip(p[:-1], p[1:]):
            ls += math.log(prob_fn(prev, w)); n += 1
    return math.exp(-ls / n), n

# --- choose K and lambda on VALIDATION ---
k_grid   = [1, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001]
lam_grid = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

val_k   = {k: perplexity(val_sents, lambda a, b, k=k: p_addk(a, b, k))[0] for k in k_grid}
val_lam = {l: perplexity(val_sents, lambda a, b, l=l: p_interp(a, b, l))[0] for l in lam_grid}
best_k   = min(val_k,   key=val_k.get)
best_lam = min(val_lam, key=val_lam.get)

print('Validation perplexity, add-k        :', {k: round(v, 1) for k, v in val_k.items()})
print('Validation perplexity, interpolation:', {l: round(v, 1) for l, v in val_lam.items()})
print(f'\nChosen on validation: K = {best_k}, lambda = {best_lam}')

In [ ]:
# --- final evaluation on TEST (done once) ---
results = {
    'Unigram (add-one)':                          perplexity(test_sents, lambda a, b: p_unigram(b)),
    'Bigram (add-one / Laplace)':                 perplexity(test_sents, p_laplace),
    f'Bigram add-k (k={best_k})':                 perplexity(test_sents, lambda a, b: p_addk(a, b, best_k)),
    f'Bigram + unigram interpolation (λ={best_lam})': perplexity(test_sents, lambda a, b: p_interp(a, b, best_lam)),
}
n_pred = next(iter(results.values()))[1]
print(f'Test predictions evaluated: {n_pred:,}\n')
display(pd.DataFrame({'Test perplexity': {k: v[0] for k, v in results.items()}}).round(1))

**Reading the table.** With a large vocabulary and sparse data, add-one smoothing gives too much probability mass to unseen bigrams, so the plain Laplace bigram can be *worse* than the unigram baseline. A small constant `k` or interpolation with the unigram model fixes this, because most of the probability stays on bigrams that were really seen.

In [ ]:
unseen, total = 0, 0
for s in test_sents:
    p = prep(s)
    for prev, w in zip(p[:-1], p[1:]):
        total += 1
        unseen += bigram[(prev, w)] == 0
print(f'Unseen bigrams in test: {unseen:,} / {total:,} = {unseen/total:.1%}')

## 5. Summary

* **Preprocessing:** cleaning (HTML/URL/agency boilerplate) → sentence segmentation → tokenization → POS tagging → lowercasing → punctuation/number removal → stop-word removal → POS-aware lemmatization.
* **Data split:** 90 % of train builds the vocabulary and the unigram/bigram counts; 10 % of train (validation) only selects `K` and `λ`; the test set is used only for the worked unseen-sentence example and the final perplexity.
* **Markov assumption:** $P(w_i \mid w_1..w_{i-1}) \approx P(w_i \mid w_{i-1})$, so $P(sentence) \approx \prod P(w_i \mid w_{i-1})$ with `<s>`/`</s>` padding.
* **Smoothing:** without smoothing, any unseen bigram makes the whole sentence probability 0; add-one, add-k and interpolation are compared by perplexity.
* **Note:** stop words are removed *before* building the bigrams (as the assignment requires), so two words that are neighbours in the model were not always neighbours in the original sentence.